# Standard Stimulus Creation

This notebook helps you build stimuli that are reliable, easy to analyse with the rest of
the pipeline, and that carry the checks needed to trust the recording.

## What are a `.bin` and a `.vec`?

A visual stimulus on our rigs is described by **two files**:

- the **`.bin`** is a **table of images**: all the frames the projector (DMD) can show, one
  after the other, each identified by its index (frame 0, frame 1, ...). It says nothing
  about *when* they are shown.
- the **`.vec`** is the **sequence**: one row per projector refresh (= one trigger recorded
  with the MEA data), saying which frame of the `.bin` to show at that moment, plus a
  **sequence key** (its last column) that tells the analysis which stimulus sequence and
  repetition this moment belongs to. The other columns hold rig-specific settings you can
  usually leave at 0.

So the `.bin` is the image library and the `.vec` is the playlist. Showing the same frame
for 10 rows of the vec keeps it on screen for 10 refreshes; the same frame can be reused as
often as you want. On the analysis side, the triggers give the time of each vec row, and the
sequence keys let the pipeline cut the spikes into sequences and repetitions.

In [ ]:
%reload_ext autoreload
%autoreload 2

# standard packages
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# project packages
import params
import utils

## I— How to create a vec file with the right sequence keys

This pipeline needs one thing from your vec file: its last column must give a *sequence key* to every trigger. The other columns are ignored here.

### The key structure

Each key is a whole number read as two parts:

```
<sequence-type digits><repetition digits>
                      └── the last n_digit_for_rep digits (4 by default)
```

- **Sequence type** (leading digits): *what* was shown — e.g. grating direction, image id, condition number.
- **Repetition** (last `n_digit_for_rep` digits): *which presentation* of that sequence type, zero-padded.

Example with `n_digit_for_rep = 4`, sequence type = grating direction (1–8):

| Direction | Repetition | Key |
|-----------|-----------|-----|
| 1 | 0 | `10000` |
| 1 | 1 | `10001` |
| 2 | 0 | `20000` |
| 8 | 3 | `80003` |

### The rules

1. **One row per trigger.** The very first line of the file is treated as a header and dropped, so it must *not* be a real trigger.
2. **Same key for every trigger of one repetition.** All triggers making up direction 1, repetition 0 carry the key `10000`.
3. **Start sequence types at 1, not 0.** Keys are stored as whole numbers, so a leading zero is lost (`01000` becomes `1000`). The first digit must be non-zero.
4. **Number repetitions from 0**, zero-padded to `n_digit_for_rep` digits. Repetition `0000` of each sequence type **must exist** — it is used as the timing reference.
5. **Keep every key contiguous.** All triggers sharing a key must form a single uninterrupted block in the file. Never reuse the same key for triggers that appear in two separate places — they would be merged into one (much-too-long) repetition.
6. **Use the same total width for every key** (e.g. always 5 digits) so the split between sequence type and repetition stays consistent.

Repetitions do **not** need to be in order: you can interleave sequence types and repetitions however your stimulus did, as long as each key stays a single contiguous block (rule 5).

### An example to copy

`ResourcesAndTools/StimMaking/add_standard_keys_to_dg_vec.ipynb` builds exactly these keys for an 8-direction, 4-repetition drifting-grating stimulus (each repetition is 12 s = 600 triggers at 50 Hz). Use it as a template: it reads a raw vec, writes the sequence key into the last column following the rules above, and saves the result.

## II — Generating a stimulus `.bin` file

A stimulus is described by **two** files that work as a pair:

| file | what it holds |
|------|---------------|
| `.vec` | one row per trigger; column 1 says **which frame** of the `.bin` to show (see Appendix A for the last column) |
| `.bin` | the **frames themselves** — the actual images sent to the DMD |

So the `.vec` is a playlist and the `.bin` is the image library. This appendix is about writing the `.bin`.

### The `BinFile` helper

Reading and writing `.bin` files goes through `BinFile`:

```python
from utils.binfile import BinFile     # note: not re-exported as utils.BinFile
```

- `BinFile.read_header(path)` → `{'xsize', 'ysize', 'nb_images', 'nb_bits'}` without opening the whole file.
- **Reading**: `BinFile(path, 0, 0, rig_id, mode="r")`, then `read_frame(i)` gives a 2-D array of floats in `[0, 1]` (the frame size comes from the header, which is why the two zeros are ignored). `len(obj)` / `obj.nb_frames` give the frame count.
- **Writing**: `BinFile(path, xsize, ysize, rig_id, nb_images=N, mode="w")`, then `append(frame)` for each frame (again floats in `[0, 1]`), then `close()`.

Two things to note:

1. **`nb_images` must be the final frame count.** The header is written when the file is opened, so it cannot be corrected afterwards. Appending a different number of frames leaves a file whose header lies about its own length. [In practive this is not read by the rig but is helphful for consistency.]
2. **`rig_id` matters.** Each rig has its own display polarity and optical path, and frames are stored *pre-corrected* for it. `BinFile` applies that correction automatically from `params.rig_params` — so write with the rig you will actually display on (`params.MEA`). Reading a file back with the same `rig_id` returns the image you started from.

### The `F` test — your own orientation check

Before anything else, every stimulus-making code should be checked with a **letter `F`**, as long as it is not full field stimulation. The F
is strongly asymmetric, so *any* unwanted flip or rotation of the display is immediately
visible — unlike a circle or a checkerboard, which look fine mirrored.

The F is **not** provided by this pipeline, and that is on purpose: its job is to debug **your
own code**, so it only means something if *your* code draws it, writes it into *your* `.bin`,
and plays it from *your* `.vec`. Concretely:

- draw an upright F with your stimulus code and add it as one frame of your `.bin`
  (anywhere you like — its index just has to appear in your vec's frame column),
- show it for a second or so at the start of your `.vec`, with the **reserved sequence key
  `9999`** in the last column (i.e. `99990000` with 4 repetition digits). That id is kept free
  by the pipeline (see section III) so it can never collide with your stimulus's own keys, and
  every analysis then knows which rows are the F test.

If the F you built with your code shows upright on the rig, your whole chain (drawing →
`.bin` → `.vec` → display) is oriented correctly.

### Check it before you record

Once the `.bin` and `.vec` exist, preview them with the **StimulusDisplayer** tool (its own repository, kept next to this pipeline). It replays the vec against the bin and shows what the DMD would display, so you can confirm the frame order, the sequence keys, the shutter/colour columns — and the F.

- Point it at your `.bin` and `.vec`, choose your MEA, and you should see an **upright F**. If it appears mirrored or rotated, the display orientation is wrong — fix that *before* recording, because it silently flips every receptive field you compute afterwards.
- `StimulusDisplayer/make_F_test.py` generates a ready-made F-only stimulus (`F_TEST_432x432.bin` + `.vec`) to check the **rig** itself. It does not replace the F made with your own code (above), which is what tests *your* chain.

You can also the following code to run a few test.


In [ ]:
from utils.binfile import BinFile

# ---- Inputs ------------------------------------------------------------
# Stimuli live in params.stim_directory (ResourcesAndTools/StandardVec). Change the name.
bin_path = os.path.join(params.stim_directory, "my_stimulus.bin")
frame_size = 432  # square stimulus window, in pixels
rig_id = params.MEA  # frames are stored pre-corrected for THIS rig

# ---- Build the frames --------------------------------------------------
frames = [
    np.full(
        (frame_size, frame_size), 0.5
    ),  # frame 0 : mid-grey (mean-luminance reference)
    # frames 1+ : add your own stimulus frames here, as arrays of floats in [0, 1]
    #             (including the F you draw with your own code, see above)
]

# ---- Write the .bin ----------------------------------------------------
# nb_images must be the FINAL number of frames (the header is written upfront).
binf = BinFile(
    bin_path, frame_size, frame_size, rig_id, nb_images=len(frames), mode="w"
)
for frame in frames:
    binf.append(frame)
binf.close()
print(f"Wrote {len(frames)} frames to {bin_path}")
print(BinFile.read_header(bin_path))

# ---- Read it back and check the frames look right ----------------------
reader = BinFile(bin_path, 0, 0, rig_id, mode="r")
fig, axs = plt.subplots(1, len(reader), figsize=(4 * len(reader), 4))
for i, ax in enumerate(np.atleast_1d(axs)):
    ax.imshow(reader.read_frame(i), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"frame {i}", fontsize=14)
    ax.axis("off")
reader.close()
plt.show()

## III — The standard preamble: grey and the four squares

Every stimulus should begin with the same short **preamble** so each recording carries its
own display check.

**Standard bin** — the first five frames are always:

| frame | content |
|-------|---------|
| 0 | grey (mean-luminance reference) |
| 1–4 | the four squares: top, right, bottom, left |

Your real stimulus frames then start at frame **5**.

**Standard vec** — always play the **four squares** *before* the real stimulus. The squares
are flashed one at a time just outside the MEA (top, right, bottom, left); analysed in
`A_Standard_Vec_Analysis.ipynb` they check that the display is spatially consistent — a cell
whose receptive field is at the top responds most to the top square.

**Reserved sequence ids** — the preamble uses ids in the `999x` block so they never collide
with a real stimulus's ids (which start at 1): squares `9991` (top), `9992` (right), `9993`
(bottom), `9994` (left). `9999` is reserved too, for the **F test you make with your own
code** (section II) — the pipeline never generates it. To prepend the preamble to a real
stimulus, keep the stimulus's own `1..N` ids and **offset its frame indices by 5** (the five
standard frames come first in the bin).

The cell below writes this preamble as its own stimulus (`standard_preamble.bin` +
`_std.vec`) — run it as a standalone check recording, or use it as the block to prepend to
your stimuli.

In [ ]:
# ---- parameters --------------------------------------------------------
rig_id = params.MEA
frame_size = params.get_rig_params(rig_id)["size_dmd"][0]  # square DMD frame, in pixels
refresh_hz = 40  # DMD refresh you display it at (triggers per second)
square_duration_s = 1.0  # how long each square is shown
n_square_reps = 20  # repetitions of the 4-square cycle

# ---- frames: grey and the four squares ---------------------------------
frames = utils.standard_bin_frames(
    frame_size, mea=rig_id
)  # [grey, top, right, bottom, left]
bin_path = os.path.join(params.stim_directory, "standard_preamble.bin")
binf = BinFile(
    bin_path, frame_size, frame_size, rig_id, nb_images=len(frames), mode="w"
)
for frame in frames:
    binf.append(frame)
binf.close()
print(f"Wrote {len(frames)} frames to {bin_path}")

# ---- vec: play the four squares (reserved ids) --------------------------
vec_rows = utils.standard_preamble_vec_rows(
    refresh_hz=refresh_hz,
    square_duration_s=square_duration_s,
    n_square_reps=n_square_reps,
)
header = [
    0,
    len(vec_rows),
    0,
    0,
    0,
]  # first line = summary (col 1 = total frames), dropped on load
vec_path = os.path.join(params.stim_directory, "standard_preamble_std.vec")
np.savetxt(vec_path, np.vstack([header, vec_rows]), fmt="%d")
print(
    f"Wrote {len(vec_rows)} rows to {vec_path}  ({n_square_reps} reps of the 4 squares)"
)

# ---- preview the four squares + the MEA footprint ----------------------
fraction = utils.mea_extent_on_display(rig_id)
squares = utils.make_four_squares_frames(frame_size, fraction)
fig, axs = plt.subplots(1, len(squares), figsize=(4 * len(squares), 4))
for ax, (name, frame) in zip(axs, squares):
    ax.imshow(frame, cmap="gray", vmin=0, vmax=1, extent=[-1, 1, -1, 1])
    ax.add_patch(
        plt.Rectangle(
            (-fraction, -fraction),
            2 * fraction,
            2 * fraction,
            fill=False,
            edgecolor="red",
            lw=1.5,
        )
    )
    ax.set_title(f"{name} square", fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
axs[0].set_ylabel("red box = MEA", fontsize=12)
plt.show()

## IV — Prepend the preamble to your stimulus (the easy way)

It is recommended that every recording should begin with the preamble. Once you have generated your
stimulus's `.bin` and `_std.vec` (from your stimulus-making tool), this one call adds the
standard preamble in front and writes the ready-to-display files:

- your stimulus frames are **copied byte-for-byte** (unchanged) after the five standard frames,
- the vec plays the **four squares** (reserved ids) first, then your stimulus, with your
  stimulus's frame indices shifted and its own sequence ids untouched. The preamble keys
  are written with the **same number of repetition digits as your stimulus** (inferred from
  your vec and printed — pass `n_digit_for_rep=` if the guess is wrong), so one
  `n_digit_for_rep` decodes the whole vec in `A_Standard_Vec_Analysis`.

Your own F test (section II) stays inside your stimulus's `.bin` / `.vec` — it is carried
along unchanged, just after the squares.

Point it at your files, pick the output names, and display the result on the rig. The call
ends by **checking the written pair by content** (section V) and prints `PREAMBLE OK` — if it
prints a problem instead, do not display the files.

In [ ]:
# Your existing stimulus (from your stimulus-making tool), for THIS rig (params.MEA):
dir_path = "/home/"
stimulus_bin = os.path.join(dir_path, "exemple.bin")
stimulus_vec = os.path.join(dir_path, "exemple.vec")

# Where to write the preamble-prefixed files to display on the rig:
output_bin = os.path.join(dir_path, "exemple_with_preamble.bin")
output_vec = os.path.join(dir_path, "exemple_with_preamble_std.vec")

utils.prepend_standard_preamble(
    stimulus_bin,
    stimulus_vec,
    output_bin,
    output_vec,
    mea=params.MEA,
    n_square_reps=1,  # keep it short when prepending to every recording (a few reps = a few seconds)
)

## V — Check the '4 squares' Stim was added correctly

The frame offset between the preamble and your stimulus is the easiest thing to get wrong
(prepending by hand with the wrong number, mixing an old bin with a new vec, prepending
twice...). `utils.check_standard_preamble` verifies a pair **by content, without trusting any
assumed offset**:

- it finds where the four squares really are in the bin,
- for each square key of the vec (`9991`–`9994`) it reads the frame that key points at and
  checks it *is* that square,
- it checks your own stimulus rows all point after the standard frames and inside the bin,
- and it shows the frame each square key points at, so you can confirm by eye (the "top"
  panel must show a square at the top, etc.).

Run it on **any** pair you are about to display, whether it was made by this notebook or by
your own code.

In [ ]:
bin_to_check = output_bin  # any .bin
vec_to_check = output_vec  # and the _std.vec that goes with it

preamble_ok = utils.check_standard_preamble(bin_to_check, vec_to_check, mea=params.MEA)

## VI — Tips for natural images

Natural images are the stimulus that goes wrong most often, on two points: **luminance**
and **size**.

### 1. Normalise the luminance across the whole dataset

The retina adapts to the mean luminance it receives. If your set of images is, on average,
brighter or darker than the grey shown between them (and than the checkerboard used for the
STA), the cells are recorded in a different adaptation state and the responses are not
comparable. Normalise **across the dataset, not image by image**: compute the mean and
standard deviation over **all** the images together and bring the dataset to a
**mean of 0.5** (grey) and a standard deviation of about 0.25 (as in Goldin et al. 2022),
then clip to `[0, 1]`. Individual images stay brighter or darker than grey (that is part of
being natural), but the stimulus as a whole averages to grey.

Two pitfalls:

- **Do not rescale min–max afterwards.** A final `(x - x.min()) / (x.max() - x.min())` to
  "make sure values are in [0, 1]" moves the mean away from 0.5 again. Clip instead
  (`np.clip(x, 0, 1)`), and check how many pixels were clipped — a few percent is fine.
- **Check the result**, don't assume it: after normalising, print the dataset mean and the
  mean of each image. The cell below does that.

### 2. Control the size of what is displayed

An image is displayed at **one DMD pixel per image pixel**; nothing rescales it for you. So
its size on the retina is `image_pixels × pixel size of your rig` (`params.rig_params[MEA]
["pxl_size_dmd"]`: 3.5 µm on MEA 2, 2.5 µm on MEA 3). The same 480 px image covers
1680 µm on MEA 2 but 1200 µm on MEA 3. Decide the physical size you want first (the MEA is
~580 µm across; cover it with margin for the surround), convert to pixels for **your** rig,
and crop or resize the images to that.

Then **look at it before recording** with the `StimulusDisplayer` tool (in this repo): it replays your vec against your bin as the DMD would show it,
with the MEA footprint, so you see at once whether the image is the size you intended,
correctly oriented (your F test), and centred.

In [ ]:
# ---- Check a natural-image dataset before writing it to a .bin ------------------
# `images` : your list of 2-D arrays (all the same size), whatever their original scale.
images = []  # e.g. [np.array(Image.open(path).convert("L"), dtype=float) for path in paths]

if images:
    target_mean, target_std = 0.5, 0.25  # grey, moderate contrast (Goldin et al. 2022)

    stack = np.stack(images).astype(float)
    normalised = (stack - stack.mean()) / stack.std() * target_std + target_mean
    clipped_fraction = np.mean((normalised < 0) | (normalised > 1))
    normalised = np.clip(normalised, 0, 1)  # clip, do NOT min-max rescale

    # ---- luminance check: the DATASET mean must be 0.5; images may differ from it.
    print(
        f"dataset mean = {normalised.mean():.3f} (target {target_mean}), "
        f"std = {normalised.std():.3f}, clipped pixels = {100 * clipped_fraction:.1f}%"
    )
    print("per-image means:", np.round(normalised.mean(axis=(1, 2)), 3))

    # ---- size check: how big is one image on the retina of THIS rig?
    pxl_size_um = params.get_rig_params(params.MEA)["pxl_size_dmd"]
    n_rows, n_cols = normalised.shape[1:]
    print(
        f"image {n_cols}x{n_rows} px = {n_cols * pxl_size_um:.0f} x {n_rows * pxl_size_um:.0f} µm "
        f"on MEA {params.MEA} ({pxl_size_um} µm/px); the MEA is ~580 µm across"
    )

    fig, axs = plt.subplots(1, min(4, len(normalised)), figsize=(16, 4))
    for ax, img in zip(np.atleast_1d(axs), normalised):
        ax.imshow(
            img, cmap="gray", vmin=0, vmax=1
        )  # same scale for all: brightness differences are real
        ax.set_title(f"mean {img.mean():.2f}", fontsize=14)
        ax.axis("off")
    plt.show()
else:
    print("Fill `images` with your natural images to run the checks.")